In [1]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
from Common.Utils import save_training_results


In [2]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    # System Parameters
    NUM_VIDEOS = 50  # Reduced from 500 for demo speed
    CACHE_CAPACITY_PERCENT = 0.10  # 10% cache
    CACHE_SIZE = int(NUM_VIDEOS * CACHE_CAPACITY_PERCENT) # C
    TILES_PER_VIEWPORT = 4 # k
    
    # State/Action Dimensions
    # State: 10*C + 2 (See Source 468)
    # Action: 5*C + 1 (See Source 469)
    STATE_DIM = 10 * CACHE_SIZE + 2
    ACTION_DIM = 5 * CACHE_SIZE + 1
    
    # DRL Parameters
    GAMMA = 0.6            # Discount factor (Source 473)
    EPSILON = 0.05         # E-greedy parameter (Source 472)
    LR = 0.001             # Learning rate (Source 472)
    BUFFER_SIZE = 2000     # Experience replay buffer size (Source 473)
    BATCH_SIZE = 32        # Mini-batch size (Source 473)
    TARGET_UPDATE_FREQ = 200 # Update target every 200 steps (Source 473)
    TRAIN_EPOCHS = 100     # (Source 471)
    
    # History Windows for State Features
    H_SHORT = 300
    H_LONG = 1000
    
    # Reward Constants (PSNR in dB)
    R_BASE = 30.0    # Reward for serving Base Layer (Source 392)
    R_ENH = 10.0     # Reward for serving Enhancement Layer (Source 392)
    PENALTY = 0.0    # Cost for fetching from backhaul (implicit in lack of reward)

In [3]:
# --- 2. DEEP Q-NETWORK (Section VI & VII-B) ---
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        # 4 Fully Connected Layers (Input, 2 Hidden, Output)
        # Hidden layers have 5C+1 nodes (Source 469)
        hidden_dim = output_dim 
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        
        return self.fc3(x) # Linear activation for output (Source 470)

In [4]:
# --- 3. REPLAY BUFFER ---
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)
    
# --- 4. DRL AGENT ---
class DRLAgent:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Evaluation and Target Networks
        self.policy_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net = DQN(config.STATE_DIM, config.ACTION_DIM).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=config.LR)
        self.memory = ReplayBuffer(config.BUFFER_SIZE)
        self.steps_done = 0
        
    def select_action(self, state):
        # Epsilon-Greedy Policy (Source 313)
        if random.random() < self.config.EPSILON:
            return random.randint(0, self.config.ACTION_DIM - 1)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                q_values = self.policy_net(state_t)
                return q_values.argmax().item()

    def train(self):
        if len(self.memory) < self.config.BATCH_SIZE:
            return None # Not enough samples yet

        # Sample mini-batch (Source 328)
        states, actions, rewards, next_states, dones = self.memory.sample(self.config.BATCH_SIZE)

        state_batch = torch.FloatTensor(np.array(states)).to(self.device)
        action_batch = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        reward_batch = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_state_batch = torch.FloatTensor(np.array(next_states)).to(self.device)
        done_batch = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # Compute Q(s, a)
        curr_q_values = self.policy_net(state_batch).gather(1, action_batch)

        # Compute Max Q(s', a') from Target Net (Fixed Target Mechanism)
        next_q_values = self.target_net(next_state_batch).max(1)[0].unsqueeze(1)
        expected_q_values = reward_batch + (self.config.GAMMA * next_q_values * (1 - done_batch))

        # Loss Function (MSE) (Source 349)
        loss = nn.MSELoss()(curr_q_values, expected_q_values)

        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update Target Network periodically (Source 335)
        self.steps_done += 1
        if self.steps_done % self.config.TARGET_UPDATE_FREQ == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        return loss.item()

In [5]:
class AgentAdapter:
    """
    Wraps the provided LabEnv to make it compatible with the DQN Agent.
    Handles:
    1. Feature Extraction (converting raw Env observations to State Vectors).
    2. Action Decoding (converting DQN integer actions to Env commands).
    3. History Tracking (Short vs Long term memory).
    """
    def __init__(self, lab_env, config):
        self.env = lab_env
        self.c = config
        
        # History Tracking (Required for State Features x, y, z)
        # We need to track these externally if the LabEnv doesn't provide processed features
        self.video_history_short = deque(maxlen=self.c.H_SHORT)
        self.video_history_long = deque(maxlen=self.c.H_LONG)
        self.tile_history_short = deque(maxlen=self.c.H_SHORT)
        self.tile_history_long = deque(maxlen=self.c.H_LONG)
        
    def reset(self):
        obs, info = self.env.reset()
        
        # Clear Histories
        self.video_history_short.clear()
        self.video_history_long.clear()
        self.tile_history_short.clear()
        self.tile_history_long.clear()
        
        return obs, info

    def update_histories(self, current_request):
        vid_id = current_request['video_id']
        tile_ids = current_request['tile_ids']
        
        self.video_history_short.append(vid_id)
        self.video_history_long.append(vid_id)
        
        for t in tile_ids:
            # We track tiles as tuples (VideoID, TileID) to differentiate videos
            self.tile_history_short.append((vid_id, t))
            self.tile_history_long.append((vid_id, t))
    
    def get_state_vector(self, candidate_id, is_tile_candidate, tile_id=None):
        """
        Constructs the neural network input vector (10C + 2) from the LabCacheEngine status
        and our local history queues.
        """
        state = []
        
        # Access the current cache state from your Engine
        # Assuming env.cache_engine.get_slots() returns list of cached Video IDs
        current_cache_slots = self.env.cache_engine.get_slots() 
        current_cache_tiles = self.env.cache_engine.get_cached_tiles() # Should return list of lists
        
        # 1. Feature x: Cached Videos History (2 * C features)
        for i in range(self.c.CACHE_SIZE):
            if i < len(current_cache_slots):
                vid_id = current_cache_slots[i]
                if vid_id is None:
                    state.extend([0, 0])
                else:
                    state.extend([
                        self.video_history_short.count(vid_id),
                        self.video_history_long.count(vid_id)
                    ])
            else:
                state.extend([0, 0]) # Empty slot
                
        # 2. Feature y: Cached Tiles History (2 * k * C features)
        for i in range(self.c.CACHE_SIZE):
            if i < len(current_cache_slots):
                vid_id = current_cache_slots[i]
                # Assuming fixed K tiles per viewport for the feature vector size consistency
                for t_idx in range(self.c.TILES_PER_VIEWPORT):
                    if vid_id is not None and i < len(current_cache_tiles) and t_idx < len(current_cache_tiles[i]):
                        actual_tile_id = current_cache_tiles[i][t_idx]
                        target = (vid_id, actual_tile_id)
                        state.extend([
                            self.tile_history_short.count(target),
                            self.tile_history_long.count(target)
                        ])
                    else:
                        state.extend([0, 0])
            else:
                state.extend([0] * (2 * self.c.TILES_PER_VIEWPORT))

        # 3. Feature z: Candidate History (2 features)
        if is_tile_candidate:
            target = (candidate_id, tile_id)
            state.extend([
                self.tile_history_short.count(target),
                self.tile_history_long.count(target)
            ])
        else:
            state.extend([
                self.video_history_short.count(candidate_id),
                self.video_history_long.count(candidate_id)
            ])

        return np.array(state, dtype=np.float32)
    
    def decode_and_execute_action(self, action_idx, candidate_vid, candidate_tile=None):
        """
        Translates DQN Integer Action -> LabEnv Cache Command
        """
        cache_engine = self.env.cache_engine
        
        # Action 0: No Op
        if action_idx == 0:
            return # Do nothing
        
        # Actions 1..C: Replace Video
        elif 1 <= action_idx <= self.c.CACHE_SIZE:
            slot_idx = action_idx - 1
            # Perform the actual eviction/caching in your engine
            # You need to verify the exact method name in LabCacheEngine
            cache_engine.replace_video(slot_idx, candidate_vid)
            
        # Actions C+1...: Replace Tile
        else:
            base_idx = action_idx - (self.c.CACHE_SIZE + 1)
            slot_idx = base_idx // self.c.TILES_PER_VIEWPORT
            tile_pos = base_idx % self.c.TILES_PER_VIEWPORT
            
            if candidate_tile is not None:
                cache_engine.replace_tile(slot_idx, tile_pos, candidate_tile)

In [ ]:
def train_integrated_agent(adapter, agent, cfg):
    
    total_steps = 0
    episode_rewards = []
    episode_losses = []

    obs, info = adapter.reset()
    done = False
    episode_reward = 0
    episode_loss = 0
    step_count = 0
    
    n_gops = adapter.env.users_env.video_gops
    for step in count():
        
        reqs_state = info['users_requests']
        active_users = [
            (req['u'], req['video'], req['gop']) 
            for req in reqs_state if req['gop'] < n_gops
        ]
    
        if len(info['users_requests']) == 0:
            obs, info = adapter.env.step(None)
            continue
        
        current_request = info['users_requests']
        
        video = current_request['video']
        
        state_vid = adapter.get_state_vector(
            video, 
            is_tile_candidate=False
        )
        
        if not adapter.env.cache_engine.is_video_cached(video):
            pass
        
        if done:
            break
        
        
        # print(f"Step {step}, Active Users: {len(active_users)}")
        # print(f"Request State: {reqs_state}")
        # print(f"Action: {actions}")
        # print(f"Next Request State: {reqs_next_state}")
        # print(
        #     f"Reward: {rewards}, "
        #     f"Cache Hits: {enhanced_layer_cache_hits + base_layer_cache_hits}, "
        #     f"Cache Misses: {enhanced_layer_cache_misses + base_layer_cache_misses}"
        # )
        print("-----")
        
        print(f"Epoch {step+1}/{cfg.TRAIN_EPOCHS}, Reward: {episode_reward:.2f}, Loss: {episode_losses[-1]:.4f}")

In [ ]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")
    
    n_episodes = 300
    n_nodes = 3
    n_users = 200
    step_size = 10.00
    arrival_rate = 10.0  # users per second
    alpha = 0.5
    n_videos = 500
    n_gops = 30
    n_layers = 2
    n = 4
    m = 3
    n_tiles = n * m
    max_capacity = 500e6  # 500 MB
    
    
    # CPT parameters
    theta = 0.5
    lam = 3.7183

    # 1. Load Configuration
    cfg = Config()
    
    # 2. Initialize Environment
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_users=n_users,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles,
        n_gops=n_gops,
        cache_capacity=max_capacity,
        # policy=SvcLruPolicy(max_size=max_capacity)
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n_tiles,
        n=n,
        m=m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=arrival_rate,
        alpha=alpha
    )
    
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=theta,
        lam=lam
    )
    
    # 3. Initialize Agent Adapter
    adapter = AgentAdapter(env, cfg)
    
    # 4. Initialize DRL Agent
    agent = DRLAgent(cfg)
    
    # 5. Training Loop
    train_integrated_agent(adapter, agent, cfg)
    
    print("--- Training Completed ---")

--- Starting DRL Caching System ---


KeyboardInterrupt: 